# Lab 15-EDA: คู่มือ Data Understanding, Visualization, EDA เจาะลึก, Hypothesis Testing และการเลือกโมเดล
> Week 15 | CLO1–CLO4 | ต่อยอดจาก Week 1–14 ทั้งหมด | ใช้ประกอบ Final Project

สมุดนี้เป็น**คู่มือฉบับละเอียด**สำหรับขั้นตอนที่สำคัญที่สุดของ Final Project — สิ่งที่ต้องทำ**หลังจาก**เตรียมข้อมูล (data cleaning) เสร็จแล้ว แต่**ก่อน**เริ่ม fit model จริง เป้าหมายหลักของสมุดนี้คือให้นักศึกษาเข้าใจและฝึกทำ **Data Understanding**, **Data Visualization**, และ **EDA (Exploratory Data Analysis)** อย่างเป็นระบบและละเอียด ซึ่งเป็นขั้นตอนที่มักถูกมองข้ามแต่ส่งผลต่อคุณภาพของทั้ง pipeline มากที่สุด นอกจากนี้เราจะไล่ดูต่อไปถึงการตั้งสมมติฐานจากสิ่งที่เห็นใน EDA แล้วพิสูจน์ด้วย Hypothesis Testing (Week 7), การตั้งโจทย์ (Problem Framing) ให้ตรงกับประเภทข้อมูลที่มี และสุดท้ายคือกรอบการตัดสินใจเลือกโมเดล/วิธีที่เหมาะสมจากทุกวิธีที่เรียนมาในวิชานี้ (Week 8–14) เราจะใช้ **Titanic dataset** เป็นตัวอย่างตลอดทั้งเล่ม เพราะมีทั้งตัวแปร continuous และ categorical ปนกัน มี missing values จริง และมี target เป็น binary classification ที่เหมาะกับการสาธิตทุกแนวคิด ทักษะในสมุดนี้คือสิ่งที่ Data Scientist มืออาชีพใช้เวลามากที่สุดในทุกโปรเจกต์จริง (มักกล่าวกันว่า 60–80% ของเวลาทั้งโปรเจกต์คือ EDA + data preparation ไม่ใช่การ fit model)

## เนื้อหาในสมุดนี้

| Part | หัวข้อ | เชื่อมกับ |
|------|--------|----------|
| 0 | โหลดข้อมูลและมองภาพรวมแรก | — |
| 1 | **Data Understanding** — โครงสร้าง, missing values, duplicates, cardinality | Week 1 (Data types) |
| 2 | **Data Visualization เจาะลึก** — univariate, bivariate, target-focused | Week 4, Week 7 (EDA) |
| 3 | **EDA เจาะลึก** — outliers, distribution shape, multicollinearity | Week 6–7, Week 10 (VIF) |
| 4 | **ตั้งสมมติฐานและพิสูจน์** — Hypothesis Testing จาก EDA | Week 7 (CI, Hypothesis Testing) |
| 5 | **ตั้งโจทย์ (Problem Framing)** — กำหนด X, Y, task type, metric | Week 5 (Regression vs Classification) |
| 6 | **เลือกโมเดลและวิธีที่เหมาะสม** — decision framework + fit + CV | Week 8–14 ทั้งหมด |

**ก่อนเริ่ม**: สมุดนี้ใช้กับ Final Project ได้โดยตรง — แทนที่ Titanic ด้วย dataset ของกลุ่มตัวเองในแต่ละ cell แล้วทำตามโครงสร้างเดียวกัน

---
## Setup: ติดตั้งและ Import Libraries

ในส่วนนี้เราจะติดตั้ง package ที่จำเป็น (ส่วนใหญ่มีอยู่แล้วใน Google Colab) และ import libraries ที่จะใช้ตลอดทั้งเล่ม — แบ่งเป็นกลุ่มตามหน้าที่เพื่อให้เห็นภาพว่าแต่ละ library ใช้ทำอะไร

In [ ]:
# ─── ติดตั้ง packages ที่อาจไม่มีใน environment ───────────────────────
# วัตถุประสงค์: ให้ notebook รันได้ทันทีทั้งบน Google Colab และเครื่องตัวเอง
!pip install -q statsmodels scikit-learn seaborn pandas numpy scipy

In [ ]:
# ─── Import: Data handling ─────────────────────────────────────────
# วัตถุประสงค์: โหลด จัดการ และสำรวจข้อมูลเบื้องต้น
import numpy as np
import pandas as pd

# ─── Import: Visualization ──────────────────────────────────────────
# วัตถุประสงค์: สร้างกราฟสำหรับ EDA ทุกประเภท
import matplotlib.pyplot as plt
import seaborn as sns

# ─── Import: Statistical Testing (Week 7) ───────────────────────────
# วัตถุประสงค์: ทดสอบสมมติฐานที่ตั้งจาก EDA
from scipy import stats
from scipy.stats import shapiro, chi2_contingency, ttest_ind, mannwhitneyu

# ─── Import: Multicollinearity Check (Week 10) ──────────────────────
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# ─── Import: Preprocessing + Models (Week 8–12) ─────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# ─── Import: Evaluation (Week 11–14) ────────────────────────────────
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve,
                              roc_auc_score, ConfusionMatrixDisplay)

# ตั้งค่าการแสดงผลกราฟให้อ่านง่าย
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

print("พร้อมแล้ว! Libraries ทั้งหมด import สำเร็จ")

---
## Part 0: โหลดข้อมูล — Titanic Dataset

**Part นี้เราจะโหลด Titanic dataset** ผ่าน seaborn (ดึงจาก GitHub อัตโนมัติ ไม่ต้อง upload ไฟล์) เพื่อใช้เป็นตัวอย่างตลอดทั้งเล่ม เราเลือก Titanic เพราะ: (1) มีตัวแปร continuous (Age, Fare) และ categorical (Sex, Pclass, Embarked) ปนกัน เหมือน dataset จริงส่วนใหญ่ (2) มี missing values จริงให้ฝึกจัดการ (3) มี target เป็น binary (Survived) ที่ใช้สาธิต classification ได้ครบทุกวิธีที่เรียนมา — **ถ้าใช้กับ Final Project ของตัวเอง ให้แทนที่ cell นี้ด้วยการโหลด dataset ของกลุ่ม แล้วทำตามโครงสร้างเดียวกันได้เลย**

In [ ]:
# ─── โหลด Titanic Dataset จาก seaborn ────────────────────────────────
# วัตถุประสงค์: ได้ dataset ที่มีทั้ง continuous/categorical variables และ missing values จริง
df = sns.load_dataset('titanic')

# ─── สำหรับ Final Project ของตัวเอง ให้แทนที่ด้วย: ────────────────────
# df = pd.read_csv('your_dataset.csv')
# หรือถ้าใช้ ISLP dataset: from ISLP import load_data; df = load_data('ชื่อ_dataset')

print(f"Shape: {df.shape[0]} แถว, {df.shape[1]} คอลัมน์")
df.head()

In [ ]:
# ─── มองภาพรวมแรก: คอลัมน์ทั้งหมดและความหมาย ──────────────────────────
# วัตถุประสงค์: รู้จักตัวแปรทุกตัวก่อนวิเคราะห์ต่อ — ขั้นตอนแรกที่ห้ามข้าม!
column_meaning = {
    'survived': 'Target: รอดชีวิต (1) หรือไม่ (0)',
    'pclass': 'ชั้นโดยสาร (1=first, 2=second, 3=third)',
    'sex': 'เพศ',
    'age': 'อายุ (ปี) — มี missing values',
    'sibsp': 'จำนวนพี่น้อง/คู่สมรสบนเรือ',
    'parch': 'จำนวนพ่อแม่/ลูกบนเรือ',
    'fare': 'ค่าโดยสาร (ปอนด์)',
    'embarked': 'ท่าเรือที่ขึ้นเรือ (C/Q/S) — มี missing values',
    'class': 'เหมือน pclass แต่เป็น string',
    'who': 'man/woman/child',
    'deck': 'ดาดฟ้าที่พัก — missing values เยอะมาก',
    'alone': 'เดินทางคนเดียวหรือไม่',
}
for col, meaning in column_meaning.items():
    if col in df.columns:
        print(f"  {col:12s}: {meaning}")

---
## Part 1: Data Understanding — ทำความเข้าใจโครงสร้างข้อมูล

**Part นี้เราจะสำรวจ "รูปร่าง" ของข้อมูลก่อนวิเคราะห์เชิงลึก** เพื่อรู้ว่ามีปัญหาอะไรซ่อนอยู่บ้าง (missing values, duplicates, ประเภทข้อมูลผิด) ก่อนที่จะเสียเวลาวิเคราะห์ต่อบนข้อมูลที่มีปัญหา **หลักการสำคัญ**: ห้าม fit model ก่อนเข้าใจข้อมูลอย่างถ่องแท้ — เพราะ garbage in, garbage out

### 1.1 โครงสร้างพื้นฐาน: Shape, Data Types, Memory

In [ ]:
# ─── ตรวจสอบ dtypes และ memory usage ─────────────────────────────────
# วัตถุประสงค์: รู้ว่าแต่ละคอลัมน์เก็บข้อมูลประเภทไหน (numeric/categorical/boolean)
# dtype ที่ผิด (เช่น ตัวเลขที่ควรเป็น category ถูกอ่านเป็น int) จะทำให้วิเคราะห์ผิดทันที
print("=== Data Types ===")
print(df.dtypes)
print(f"\n=== Memory Usage ===")
print(f"Total: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

# แยกคอลัมน์ตามประเภทให้ใช้งานง่ายในขั้นตอนถัดไป
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

### 1.2 Missing Values — ต้องรู้ก่อนทำอะไรทั้งหมด

Missing values เป็นปัญหาที่พบบ่อยที่สุดในข้อมูลจริง และถ้าไม่จัดการให้ถูกวิธี จะทำให้ทั้ง visualization, hypothesis test, และ model ผิดพลาดหรือ error ทั้งหมด เราจะดูทั้งจำนวน สัดส่วน และ pattern ของ missing values

In [ ]:
# ─── นับ missing values ทุกคอลัมน์ ────────────────────────────────────
# วัตถุประสงค์: รู้ว่าคอลัมน์ไหนมีปัญหามากแค่ไหน ก่อนตัดสินใจว่าจะ impute หรือทิ้ง
missing = pd.DataFrame({
    'จำนวน_missing': df.isnull().sum(),
    'เปอร์เซ็นต์': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['จำนวน_missing'] > 0].sort_values('เปอร์เซ็นต์', ascending=False)
print(missing)

# ─── กฎง่ายๆ ในการตัดสินใจ ────────────────────────────────────────────
# < 5%    : impute ได้เลย (mean/median/mode) โดยไม่กระทบผลมาก
# 5-40%   : impute แบบระมัดระวัง หรือสร้าง flag column บอกว่า missing
# > 40%   : พิจารณาตัดคอลัมน์ทิ้ง เพราะข้อมูลไม่พอจะเชื่อถือได้
for col, row in missing.iterrows():
    pct = row['เปอร์เซ็นต์']
    if pct < 5:
        action = "impute ได้เลย"
    elif pct < 40:
        action = "impute อย่างระมัดระวัง / พิจารณา flag column"
    else:
        action = "พิจารณาตัดทิ้ง (missing เยอะเกินไป)"
    print(f"  {col:12s} ({pct:5.1f}%): แนะนำ → {action}")

In [ ]:
# ─── Visualize Missing Value Pattern ──────────────────────────────────
# วัตถุประสงค์: ดูว่า missing values กระจายแบบสุ่ม หรือมี pattern (เช่น หายไปพร้อมกันเป็นกลุ่ม)
# ถ้าเห็น pattern ชัดเจน อาจแปลว่า missing ไม่ใช่ Missing Completely At Random (MCAR)
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False, ax=ax)
ax.set_title('Missing Value Pattern (แถบสีเหลือง = missing)')
plt.tight_layout(); plt.show()

### 1.3 Duplicates และ Cardinality ของ Categorical Variables

In [ ]:
# ─── ตรวจสอบแถวซ้ำ ────────────────────────────────────────────────────
# วัตถุประสงค์: ข้อมูลซ้ำจะทำให้สถิติ (mean, correlation) เพี้ยนและโมเดล overfit ต่อแถวที่ซ้ำ
n_dup = df.duplicated().sum()
print(f"จำนวนแถวที่ซ้ำกันทั้งหมด: {n_dup} จาก {len(df)} แถว ({n_dup/len(df)*100:.2f}%)")

# ─── Cardinality: แต่ละ categorical column มีกี่ค่าที่ไม่ซ้ำ ─────────
# วัตถุประสงค์: cardinality สูงมาก (เช่น ID, ชื่อ) ไม่เหมาะเป็น feature ตรงๆ
# cardinality ต่ำ (2-10 categories) เหมาะทำ one-hot encoding
print("\n=== Cardinality ของ Categorical Columns ===")
for col in categorical_cols:
    n_unique = df[col].nunique()
    print(f"  {col:12s}: {n_unique:3d} unique values → {df[col].value_counts().to_dict()}")

**Insight จาก Part 1**: บันทึกสิ่งที่พบเพื่อใช้ตัดสินใจในขั้นตอนถัดไป — เช่น "age missing 19.9% ต้อง impute", "deck missing เกิน 77% ควรตัดทิ้งหรือสร้างเป็น flag แทน", "embarked missing แค่ 0.2% impute ด้วย mode ได้เลย"

---
## Part 2: Data Visualization เจาะลึก

**Part นี้คือหัวใจของสมุดนี้** — เราจะสร้างกราฟหลายมุมมองเพื่อ "เห็น" ข้อมูลก่อนวิเคราะห์เชิงตัวเลข เพราะสมองมนุษย์จับ pattern จากภาพได้เร็วกว่าตาราง (Anscombe's Quartet คือตัวอย่างคลาสสิกที่แสดงว่าข้อมูล 4 ชุดที่มี mean/variance/correlation เหมือนกันทุกประการ สามารถมีรูปร่างต่างกันโดยสิ้นเชิงเมื่อ plot) เราจะทำ 4 ระดับ: **Univariate** (มองทีละตัวแปร) → **Bivariate** (มองความสัมพันธ์ 2 ตัวแปร) → **Target-focused** (มองว่าแต่ละ feature สัมพันธ์กับ target อย่างไร) → **Multivariate** (มองหลายตัวแปรพร้อมกัน)

### 2.1 Univariate: การกระจายตัวของแต่ละตัวแปร (Numeric)

In [ ]:
# ─── Histogram + KDE ของทุก numeric column ───────────────────────────
# วัตถุประสงค์: ดูรูปร่างการกระจาย (normal? skewed? bimodal?) ของแต่ละตัวแปร
# ก่อนเลือกวิธี imputation, transformation, หรือ hypothesis test ที่เหมาะสม
num_plot_cols = ['age', 'fare', 'sibsp', 'parch']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, num_plot_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution of {col}')
    skew_val = df[col].skew()
    ax.text(0.7, 0.9, f'skew={skew_val:.2f}', transform=ax.transAxes)
plt.suptitle('Univariate Distributions — Numeric Features', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ─── Boxplot ของทุก numeric column: ดู outliers เบื้องต้น ────────────
# วัตถุประสงค์: boxplot แสดง median, IQR, และจุดที่อยู่นอก whisker (potential outliers)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, num_plot_cols):
    sns.boxplot(y=df[col], ax=ax, color='lightcoral')
    ax.set_title(col)
plt.suptitle('Boxplots — มองหา Outliers เบื้องต้น', y=1.05)
plt.tight_layout(); plt.show()

### 2.2 Univariate: สัดส่วนของแต่ละตัวแปร (Categorical)

In [ ]:
# ─── Bar chart ของทุก categorical column ─────────────────────────────
# วัตถุประสงค์: ดูว่าแต่ละ category มีสัดส่วนเท่าไร — category ที่หายาก (rare)
# อาจต้องรวมกลุ่มหรือระวังตอนแบ่ง train/test (อาจไม่มีใน fold บางอัน)
cat_plot_cols = ['pclass', 'sex', 'embarked', 'who']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, cat_plot_cols):
    df[col].value_counts().plot(kind='bar', ax=ax, color='mediumseagreen')
    ax.set_title(col); ax.tick_params(axis='x', rotation=0)
plt.suptitle('Univariate Distributions — Categorical Features', y=1.05)
plt.tight_layout(); plt.show()

### 2.3 Bivariate: ความสัมพันธ์ระหว่างตัวแปร Numeric (Correlation Heatmap)

In [ ]:
# ─── Correlation Heatmap ────────────────────────────────────────────
# วัตถุประสงค์: ดูว่าตัวแปรไหน correlate กันสูง (อาจเกิด multicollinearity — Week 10)
# และตัวแปรไหน correlate กับ target (survived) มาก — บอกใบ้ว่า feature ไหนสำคัญ
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap — Numeric Features')
plt.tight_layout(); plt.show()

print("Correlation กับ target (survived), เรียงจากสูงไปต่ำ:")
print(corr['survived'].drop('survived').sort_values(ascending=False))

### 2.4 Bivariate: Pairplot — มองหลายคู่พร้อมกัน

In [ ]:
# ─── Pairplot: scatter matrix ของ numeric features แยกสี by target ───
# วัตถุประสงค์: เห็น pattern การแยกกลุ่มของ survived=0 vs survived=1 ในหลายมิติพร้อมกัน
# ในเซลล์เดียว — เร็วกว่าการ plot ทีละคู่ด้วยมือ
sns.pairplot(df[num_plot_cols + ['survived']].dropna(),
             hue='survived', palette={0: 'crimson', 1: 'steelblue'},
             diag_kind='kde', height=2.2)
plt.suptitle('Pairplot: Numeric Features colored by Survived', y=1.02)
plt.show()

### 2.5 Target-Focused: แต่ละ Feature สัมพันธ์กับ Target อย่างไร

นี่คือขั้นตอนที่สำคัญที่สุดของ visualization สำหรับงาน classification — เราต้องการรู้ว่า feature ไหน "แยก" กลุ่ม target ได้ดี ก่อนไปทำ hypothesis test และเลือกโมเดล

In [ ]:
# ─── Numeric feature vs Target: Boxplot แยกกลุ่ม ─────────────────────
# วัตถุประสงค์: ถ้า boxplot ของสอง class ไม่ทับกันมาก = feature นี้ discriminative
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ['age', 'fare']):
    sns.boxplot(data=df, x='survived', y=col, ax=ax, palette=['crimson', 'steelblue'])
    ax.set_title(f'{col} by Survived')
    ax.set_xticklabels(['No (0)', 'Yes (1)'])
plt.tight_layout(); plt.show()

In [ ]:
# ─── Categorical feature vs Target: Stacked/Grouped Bar (สัดส่วน) ────
# วัตถุประสงค์: ดูว่า category ไหนมี survival rate สูง/ต่ำผิดปกติ
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['pclass', 'sex', 'embarked']):
    survival_rate = df.groupby(col)['survived'].mean().sort_values(ascending=False)
    survival_rate.plot(kind='bar', ax=ax, color='darkorange')
    ax.set_title(f'Survival Rate by {col}')
    ax.set_ylabel('P(Survived)')
    ax.axhline(df['survived'].mean(), color='gray', linestyle='--',
               label=f'Overall = {df["survived"].mean():.2f}')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("สังเกต: ถ้าเส้น bar สูง/ต่ำกว่าเส้น overall มาก = feature นี้น่าจะมีผลจริง")
print("→ ต้องพิสูจน์ด้วย Hypothesis Test ใน Part 4 ก่อนสรุป (ไม่ใช่แค่ 'ดูด้วยตา')")

### 2.6 Multivariate: มองสามตัวแปรพร้อมกัน (FacetGrid)

In [ ]:
# ─── FacetGrid: Age distribution แยกตาม Pclass และ Survived พร้อมกัน ─
# วัตถุประสงค์: บางครั้ง effect ของตัวแปรหนึ่งขึ้นกับตัวแปรอื่น (interaction — Week 10)
# การดูแค่ 2 มิติอาจพลาด pattern ที่ซับซ้อนกว่านั้น
g = sns.FacetGrid(df, col='pclass', hue='survived', height=3.5,
                   palette={0: 'crimson', 1: 'steelblue'})
g.map(sns.histplot, 'age', kde=True, alpha=0.5)
g.add_legend(title='Survived')
g.fig.suptitle('Age Distribution by Pclass และ Survived', y=1.05)
plt.show()

**Insight จาก Part 2**: จากกราฟทั้งหมด ควรสรุปเป็นข้อความสั้นๆ เช่น "Sex และ Pclass ดูมีผลต่อ Survived ชัดเจนมาก (survival rate ต่างกันมาก), Age มีผลปานกลาง (คนอายุน้อยรอดมากกว่าเล็กน้อย), Fare correlate กับ Pclass สูง (อาจเกิด multicollinearity)" — insight เหล่านี้จะถูกนำไปพิสูจน์อย่างเป็นทางการใน Part 4

---
## Part 3: EDA เจาะลึก (Deep-Dive Analysis)

**Part นี้เราจะวิเคราะห์เชิงลึกกว่า Part 2** — ไม่ใช่แค่ "ดู" กราฟ แต่ใช้เกณฑ์เชิงตัวเลขที่ชัดเจนในการตัดสินใจ: Outlier คือจุดไหนกันแน่ (ไม่ใช่แค่ "ดูเหมือน outlier")? การกระจายตัวเบ้ (skewed) แค่ไหนถึงต้อง transform? Feature ไหน multicollinear กันจนต้องตัดออก? คำตอบเหล่านี้จะกำหนดว่าเราต้อง preprocess ข้อมูลอย่างไรก่อนเข้าโมเดล

### 3.1 Outlier Detection ด้วย IQR Method

In [ ]:
# ─── หา Outliers ด้วย IQR Rule ────────────────────────────────────────
# วัตถุประสงค์: ระบุ outlier อย่างเป็นระบบด้วยเกณฑ์คณิตศาสตร์ที่ชัดเจน
# กฎ: จุดที่อยู่นอกช่วง [Q1 - 1.5*IQR, Q3 + 1.5*IQR] ถือเป็น outlier
def detect_outliers_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = series[(series < lower) | (series > upper)]
    return outliers, lower, upper

for col in ['age', 'fare']:
    outliers, lower, upper = detect_outliers_iqr(df[col].dropna())
    pct = len(outliers) / df[col].notna().sum() * 100
    print(f"{col:6s}: ช่วงปกติ [{lower:.1f}, {upper:.1f}] | "
          f"outliers = {len(outliers)} จุด ({pct:.1f}%)")

# ─── การตัดสินใจ ──────────────────────────────────────────────────────
# outlier ไม่ได้แปลว่า "ข้อมูลผิด" เสมอไป — Fare ที่สูงมากอาจเป็นตั๋ว suite ของจริง
# ควรตรวจสอบที่มาก่อนตัดสินใจ ลบ, cap (winsorize), หรือเก็บไว้ตามเดิม

### 3.2 รูปร่างการกระจายตัว: Skewness, Kurtosis, QQ Plot, Normality Test

In [ ]:
# ─── Skewness และ Kurtosis ────────────────────────────────────────────
# วัตถุประสงค์: วัดความเบ้ (skewness) และความแหลม (kurtosis) เชิงตัวเลข
# |skew| > 1 = เบ้มาก (อาจต้อง log-transform ก่อนใช้ในโมเดลที่ assume normality เช่น LDA)
print("=== Skewness และ Kurtosis ===")
for col in ['age', 'fare']:
    s = df[col].dropna()
    print(f"{col:6s}: skewness = {s.skew():.3f}, kurtosis = {s.kurt():.3f}")

print("\nFare มี skewness สูงมาก (เบ้ขวา) — ลองดู QQ Plot และทดสอบ normality ต่อ")

In [ ]:
# ─── QQ Plot: เทียบ distribution จริงกับ Normal Distribution ─────────
# วัตถุประสงค์: ถ้าจุดเรียงตามเส้นทแยงมุม = ใกล้เคียง Normal
# ถ้าจุดโค้งออกจากเส้น (โดยเฉพาะปลายทั้งสองข้าง) = ไม่ Normal
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
stats.probplot(df['fare'].dropna(), dist="norm", plot=axes[0])
axes[0].set_title('QQ Plot: Fare (ก่อน transform) — เบ้ขวามาก')

stats.probplot(np.log1p(df['fare'].dropna()), dist="norm", plot=axes[1])
axes[1].set_title('QQ Plot: log(1+Fare) — ใกล้ Normal มากขึ้น')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Shapiro-Wilk Test: ทดสอบ Normality อย่างเป็นทางการ (Week 7) ─────
# วัตถุประสงค์: ยืนยันด้วยตัวเลข ไม่ใช่แค่ดู QQ plot ด้วยตา
# H0: ข้อมูลมาจาก Normal Distribution | H1: ไม่ได้มาจาก Normal Distribution
# หมายเหตุ: Shapiro-Wilk แม่นกับ n ไม่ใหญ่มาก ถ้า n > 5000 ควรดู QQ plot ประกอบ
sample = df['fare'].dropna().sample(min(500, df['fare'].notna().sum()), random_state=42)
stat, p_value = shapiro(sample)
print(f"Shapiro-Wilk test บน Fare: statistic={stat:.4f}, p-value={p_value:.2e}")
if p_value < 0.05:
    print("→ Reject H0: Fare ไม่ได้มาจาก Normal Distribution (p < 0.05)")
    print("→ ผลกระทบ: ถ้าจะใช้ LDA (assume Gaussian) ต้อง transform ก่อน, "
          "หรือถ้าจะทำ t-test ต้องพิจารณา non-parametric alternative (Mann-Whitney U)")
else:
    print("→ ไม่มีหลักฐานเพียงพอที่จะปฏิเสธว่าเป็น Normal Distribution")

### 3.3 Multicollinearity Check ด้วย VIF (เชื่อมกับ Week 10)

In [ ]:
# ─── Variance Inflation Factor (VIF) ─────────────────────────────────
# วัตถุประสงค์: ตรวจว่า numeric features ตัวไหน correlate กันสูงจนอาจทำให้
# coefficient ของ regression/logistic ไม่เสถียร (เชื่อมกับ Week 10: Multicollinearity)
# กฎทั่วไป: VIF > 5 = เริ่มน่ากังวล, VIF > 10 = multicollinearity รุนแรง
vif_data = df[['age', 'fare', 'sibsp', 'parch']].dropna()
vif_df = pd.DataFrame()
vif_df['feature'] = vif_data.columns
vif_df['VIF'] = [variance_inflation_factor(vif_data.values, i)
                  for i in range(vif_data.shape[1])]
print(vif_df.round(2))
print("\nทุกตัวมี VIF ต่ำ → ไม่มี multicollinearity รุนแรงในกลุ่ม numeric features นี้")

### 3.4 สรุป EDA: ตัดสินใจ Preprocessing Strategy

จากทุกสิ่งที่พบใน Part 1–3 เราสามารถวางแผน preprocessing ได้อย่างมีเหตุผล (ไม่ใช่สุ่มทำ):
- **Age** (missing 19.9%, เบ้เล็กน้อย): impute ด้วย median
- **Embarked** (missing 0.2%): impute ด้วย mode
- **Deck** (missing >75%): ตัดคอลัมน์ทิ้ง หรือสร้าง flag `has_deck_info`
- **Fare** (เบ้ขวามาก, ไม่ Normal): พิจารณา log-transform ถ้าจะใช้กับ LDA/Logistic
- **VIF ต่ำทุกตัว**: ไม่ต้องตัด feature ออกเพราะ multicollinearity

---
## Part 4: ตั้งสมมติฐานจาก EDA และพิสูจน์ด้วย Hypothesis Testing (Week 7)

**Part นี้เราจะนำ insight จาก Part 2 มาพิสูจน์อย่างเป็นทางการ** — การ "ดูกราฟแล้วรู้สึกว่ามีผล" ไม่เพียงพอสำหรับข้อสรุปทางวิทยาศาสตร์ ต้องตั้ง H₀/H₁ แล้วทดสอบด้วยสถิติที่เหมาะสมกับประเภทตัวแปร: **Chi-square test** สำหรับ categorical-vs-categorical, **t-test/Mann-Whitney** สำหรับ numeric-vs-binary-group

### 4.1 สมมติฐานที่ 1: Pclass มีผลต่อ Survived หรือไม่? (Categorical vs Categorical → Chi-square)

In [ ]:
# ─── Chi-square Test of Independence ─────────────────────────────────
# วัตถุประสงค์: ทดสอบว่า Pclass กับ Survived independent กันหรือไม่
# H0: Pclass และ Survived เป็นอิสระต่อกัน (ไม่มีความสัมพันธ์)
# H1: Pclass และ Survived มีความสัมพันธ์กัน
contingency = pd.crosstab(df['pclass'], df['survived'])
print("Contingency Table:")
print(contingency)

chi2, p_value, dof, expected = chi2_contingency(contingency)
print(f"\nChi-square statistic = {chi2:.4f}, df = {dof}, p-value = {p_value:.2e}")
if p_value < 0.05:
    print("→ Reject H0: Pclass และ Survived มีความสัมพันธ์กันอย่างมีนัยสำคัญ (p < 0.05)")
    print("→ สรุป: Pclass เป็น feature ที่ควรใส่ในโมเดลแน่นอน")
else:
    print("→ ไม่มีหลักฐานเพียงพอที่จะปฏิเสธ H0")

### 4.2 สมมติฐานที่ 2: Sex มีผลต่อ Survived หรือไม่? (Categorical vs Categorical → Chi-square)

In [ ]:
# ─── Chi-square Test: Sex vs Survived ────────────────────────────────
# วัตถุประสงค์: ทดสอบ "women and children first" policy ที่เห็นเป็น pattern ใน Part 2
contingency_sex = pd.crosstab(df['sex'], df['survived'])
chi2, p_value, dof, expected = chi2_contingency(contingency_sex)
print(f"Contingency Table:\n{contingency_sex}\n")
print(f"Chi-square = {chi2:.4f}, p-value = {p_value:.2e}")
print("→ Reject H0" if p_value < 0.05 else "→ ไม่ reject H0",
      "— Sex", "มีผล" if p_value < 0.05 else "ไม่มีผลชัดเจน", "ต่อ Survived")

### 4.3 สมมติฐานที่ 3: Age แตกต่างกันระหว่างกลุ่มรอด/ไม่รอดหรือไม่? (Numeric vs Binary Group → t-test)

In [ ]:
# ─── เลือก test ให้ถูกต้องตาม normality (จาก Part 3.2) ────────────────
# วัตถุประสงค์: ถ้า Age เป็น Normal ใช้ t-test (parametric); ถ้าไม่ ใช้ Mann-Whitney U (non-parametric)
age_survived = df.loc[df['survived'] == 1, 'age'].dropna()
age_died = df.loc[df['survived'] == 0, 'age'].dropna()

# ตรวจ normality ของแต่ละกลุ่มก่อนเลือก test
_, p_norm1 = shapiro(age_survived.sample(min(500, len(age_survived)), random_state=42))
_, p_norm2 = shapiro(age_died.sample(min(500, len(age_died)), random_state=42))
print(f"Normality check: survived group p={p_norm1:.3f}, died group p={p_norm2:.3f}")

if p_norm1 > 0.05 and p_norm2 > 0.05:
    # ทั้งสองกลุ่ม Normal → ใช้ Independent t-test
    stat, p_value = ttest_ind(age_survived, age_died, equal_var=False)
    test_name = "Independent t-test (Welch's)"
else:
    # ไม่ Normal → ใช้ non-parametric alternative
    stat, p_value = mannwhitneyu(age_survived, age_died, alternative='two-sided')
    test_name = "Mann-Whitney U test (non-parametric)"

print(f"\n{test_name}: statistic={stat:.4f}, p-value={p_value:.4f}")
print(f"Mean age (survived) = {age_survived.mean():.1f}, "
      f"Mean age (died) = {age_died.mean():.1f}")
print("→ Reject H0" if p_value < 0.05 else "→ ไม่ reject H0",
      "— Age", "มีผลต่อ Survived อย่างมีนัยสำคัญ" if p_value < 0.05 else "ไม่มีผลชัดเจน")

### 4.4 สรุปผล Hypothesis Testing ทั้งหมด

In [ ]:
# ─── ตารางสรุป: feature ไหนพิสูจน์แล้วว่ามีผลจริงทางสถิติ ─────────────
# วัตถุประสงค์: ใช้เป็นหลักฐานอ้างอิงตอนเลือก feature เข้าโมเดลใน Part 6
summary_tests = pd.DataFrame([
    {'Feature': 'Pclass', 'Test': 'Chi-square', 'H0': 'Independent จาก Survived'},
    {'Feature': 'Sex',    'Test': 'Chi-square', 'H0': 'Independent จาก Survived'},
    {'Feature': 'Age',    'Test': 't-test / Mann-Whitney', 'H0': 'Mean เท่ากันทั้งสองกลุ่ม'},
])
print(summary_tests.to_string(index=False))
print("\nนำผล p-value และข้อสรุปจากแต่ละ cell ด้านบนมาเติมในตารางนี้เพื่อใช้อ้างอิงตอนเลือก feature ใน Part 6")

---
## Part 5: ตั้งโจทย์ (Problem Framing)

**Part นี้เราจะแปลง insight ทั้งหมดจาก Part 1–4 ให้เป็นโจทย์ Machine Learning ที่ชัดเจน** ขั้นตอนนี้สำคัญมากเพราะการตั้งโจทย์ผิด (เช่น เลือก metric ผิด หรือกำหนด task type ผิด) จะทำให้งานทั้งหมดหลังจากนี้ผิดทิศทาง แม้โมเดลจะ fit ได้ดีแค่ไหนก็ตาม

### 5.1 กำหนด Task Type จากลักษณะของ Target (Y)

| ลักษณะของ Y | Task Type | วิธีที่ใช้ได้ในวิชานี้ |
|---|---|---|
| Continuous (ตัวเลขต่อเนื่อง เช่น ราคา, คะแนน) | **Regression** | SLR/MLR (Week 8–9), Polynomial Regression (Week 10) |
| Binary (2 กลุ่ม เช่น รอด/ไม่รอด, default/ไม่ default) | **Binary Classification** | Logistic, LDA, QDA, Naive Bayes, KNN (Week 11–12) |
| Multi-class (>2 กลุ่มไม่มีลำดับ) | **Multinomial Classification** | Multinomial Logistic, LDA, QDA, Naive Bayes, KNN (Week 11–12) |
| Count (จำนวนนับ 0,1,2,... เช่น จำนวนครั้ง) | **Count Regression** | Poisson Regression / GLM (Week 13) |

**สำหรับตัวอย่างของเรา**: `survived` เป็น binary (0/1) → **Binary Classification**

In [ ]:
# ─── กำหนด X และ Y อย่างชัดเจน ────────────────────────────────────────
# วัตถุประสงค์: ตัดสินใจว่า feature ไหนใช้ได้จริง โดยอ้างอิงจากผล EDA + Hypothesis Test
# หลักการเลือก feature:
#   1. มีความสัมพันธ์กับ target ที่พิสูจน์แล้ว (Part 4)
#   2. ไม่มี missing values มากเกินไป (Part 1) หรือถูก impute แล้ว
#   3. ไม่ leak ข้อมูลอนาคต (เช่น ห้ามใช้ตัวแปรที่รู้ได้หลังเหตุการณ์เกิดขึ้นแล้ว)

feature_cols = ['pclass', 'sex', 'age', 'fare', 'sibsp', 'parch']
target_col = 'survived'

print(f"Target (Y): {target_col} — Binary Classification")
print(f"Features (X) ที่เลือก: {feature_cols}")
print(f"เหตุผล: pclass และ sex พิสูจน์แล้วว่ามีผลจริง (Part 4.1, 4.2), "
      f"age มีผลปานกลาง (Part 4.3), fare/sibsp/parch เก็บไว้เพราะมี "
      f"ความสัมพันธ์เชิงทฤษฎีกับการเอาตัวรอด (ฐานะ/ขนาดครอบครัว)")

### 5.2 เลือก Success Metric ให้ตรงกับคำถามทางธุรกิจ/งานวิจัย

การเลือก metric ต้องตอบคำถามว่า **"ผิดพลาดแบบไหนที่เรายอมรับได้น้อยที่สุด"** (เชื่อมกับ Week 11–13):

| สถานการณ์ | Metric ที่เหมาะ | เหตุผล |
|---|---|---|
| Class สมดุล, สนใจภาพรวม | Accuracy | ตรงไปตรงมา ใช้ได้เมื่อไม่มี class ไหนสำคัญกว่า |
| Class ไม่สมดุล (imbalanced) | F1, Recall, Precision | Accuracy จะหลอกตาเมื่อ class ส่วนน้อยถูกทำนายผิดหมด |
| ต้องการเปรียบเทียบโมเดลข้าม threshold | AUC (ROC) | ไม่ขึ้นกับการเลือก threshold ใดๆ |
| ต้นทุนของ False Negative สูงมาก (เช่น พลาดคนป่วย) | Recall | ต้องจับ positive ให้ได้มากที่สุด แม้ False Positive จะเพิ่ม |

**สำหรับ Titanic**: survived มีสัดส่วนประมาณ 38% รอด / 62% ไม่รอด (ไม่ imbalanced รุนแรง) → ใช้ **Accuracy + AUC** เป็นหลัก และดู F1 ประกอบ

In [ ]:
# ─── ตรวจสอบความสมดุลของ target ก่อนเลือก metric ─────────────────────
print(df['survived'].value_counts(normalize=True).round(3))
print("\nสัดส่วนไม่ imbalanced รุนแรง (ไม่ถึง 90/10) → Accuracy ยังใช้ได้ "
      "แต่ควรดู AUC + F1 ประกอบเสมอเพื่อความรอบด้าน")

### 5.3 Train/Test Split (เตรียมไว้สำหรับ Part 6)

In [ ]:
# ─── เตรียมข้อมูลก่อน Split: Impute missing values ตามที่ตัดสินใจใน 3.4 ─
# วัตถุประสงค์: จัดการ missing values ก่อน split เพื่อให้ทั้ง train/test สมบูรณ์
df_model = df[feature_cols + [target_col]].copy()
df_model['age'] = df_model['age'].fillna(df_model['age'].median())

# แปลง categorical เป็นตัวเลขด้วย one-hot encoding (Week 9: dummy variables)
df_model = pd.get_dummies(df_model, columns=['sex'], drop_first=True)

X = df_model.drop(columns=[target_col])
y = df_model[target_col]

# ─── Stratified Split: รักษาสัดส่วน class เดิมใน train และ test ──────
# วัตถุประสงค์: ป้องกัน train/test มีสัดส่วน survived ต่างจากข้อมูลจริงมากเกินไป
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} แถว, Test: {X_test.shape[0]} แถว")
print(f"สัดส่วน survived — Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}")

---
## Part 6: เลือกโมเดลและวิธีที่เหมาะสม (Model & Method Selection)

**Part สุดท้ายนี้เราจะใช้กรอบการตัดสินใจที่ครอบคลุมทุกวิธีในวิชานี้** เพื่อเลือกว่าควรลองโมเดลไหนบ้าง แล้ว fit จริง เปรียบเทียบด้วย Cross-Validation (Week 14) และเลือกโมเดลสุดท้ายอย่างมีเหตุผล — ไม่ใช่แค่เลือกตัวที่ accuracy สูงสุดจาก run เดียว

### 6.1 กรอบการตัดสินใจ: ตารางสรุปทุกวิธีในวิชานี้

| Task Type | วิธี | Assumption | เมื่อไหร่ควรใช้ |
|---|---|---|---|
| **Regression** | SLR/MLR (Week 8–9) | Linearity, Normal residuals | ความสัมพันธ์เชิงเส้น, ต้องการ interpretability |
| | Polynomial Regression (Week 10) | เหมือน MLR + non-linear terms | เห็น curve ใน residual plot |
| | Poisson/GLM (Week 13) | Y เป็น count, mean=variance | Y นับจำนวน ไม่ใช่ continuous ทั่วไป |
| **Classification** | Logistic Regression (Week 11) | ไม่ assume distribution ของ X | baseline ที่ควรลองก่อนเสมอ, ต้องการ interpretability |
| | LDA (Week 12) | X ~ Gaussian, shared Σ | n เล็ก, classes แยกกันชัดเจน |
| | QDA (Week 12) | X ~ Gaussian, per-class Σ | n ใหญ่, covariance ต่างกันมากระหว่าง class |
| | Naive Bayes (Week 12) | features independent กัน | p ใหญ่มาก, features ไม่ correlate กันมาก |
| | KNN (Week 12) | ไม่ assume อะไรเลย | decision boundary ซับซ้อน, n ใหญ่พอ |

**หลักปฏิบัติ**: เริ่มด้วย **baseline ที่ง่ายที่สุดก่อนเสมอ** (Logistic สำหรับ classification, MLR สำหรับ regression) แล้วค่อยลองวิธีที่ซับซ้อนขึ้นเพื่อเทียบว่าคุ้มกับความซับซ้อนที่เพิ่มหรือไม่

In [ ]:
# ─── Fit ทุก Classifier ที่เหมาะกับปัญหานี้ (Binary Classification) ───
# วัตถุประสงค์: เปรียบเทียบทุกวิธีที่เรียนมาใน Week 11-12 อย่างเป็นระบบ
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'LDA': LinearDiscriminantAnalysis(),
    'QDA': QuadraticDiscriminantAnalysis(),
    'Naive Bayes': GaussianNB(),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
}

# ─── Pipeline: Scale ก่อนเสมอ (สำคัญมากสำหรับ KNN, LDA) ──────────────
results = []
for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    acc = (y_pred == y_test).mean()
    auc = roc_auc_score(y_test, y_proba)
    results.append({'Model': name, 'Test Accuracy': acc, 'Test AUC': auc})

results_df = pd.DataFrame(results).sort_values('Test AUC', ascending=False)
print(results_df.round(4).to_string(index=False))

### 6.2 เปรียบเทียบอย่างเป็นธรรมด้วย k-Fold Cross-Validation (Week 14)

การเปรียบเทียบด้วย test set ครั้งเดียว (ด้านบน) อาจได้ผลที่ผันผวนจาก random split — เราต้องใช้ **k-Fold CV** บน training set เพื่อได้ estimate ที่น่าเชื่อถือกว่า ก่อนตัดสินใจเลือกโมเดลสุดท้าย

In [ ]:
# ─── 5-Fold Stratified CV บน Training Set ────────────────────────────
# วัตถุประสงค์: ประเมินแต่ละโมเดลด้วยหลาย fold แทนการวัดครั้งเดียว
# ลด variance ของ estimate และให้ SE เพื่อใช้ One-SE Rule ต่อ
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='roc_auc')
    cv_results.append({
        'Model': name,
        'CV AUC (mean)': scores.mean(),
        'CV AUC (SE)': scores.std() / np.sqrt(5)
    })

cv_df = pd.DataFrame(cv_results).sort_values('CV AUC (mean)', ascending=False)
print(cv_df.round(4).to_string(index=False))

# ─── One-Standard-Error Rule (Week 14) ───────────────────────────────
# วัตถุประสงค์: เลือกโมเดลที่ "ง่ายที่สุด" ในบรรดาโมเดลที่ยัง within 1 SE ของ best
best_row = cv_df.iloc[0]
threshold = best_row['CV AUC (mean)'] - best_row['CV AUC (SE)']
within_1se = cv_df[cv_df['CV AUC (mean)'] >= threshold]
print(f"\nBest model: {best_row['Model']} (CV AUC = {best_row['CV AUC (mean)']:.4f})")
print(f"โมเดลที่อยู่ใน 1-SE ของ best (เลือกตัวที่ตีความง่ายที่สุดจากกลุ่มนี้):")
print(within_1se['Model'].tolist())

### 6.3 Final Evaluation: ประเมินโมเดลสุดท้ายบน Test Set (ครั้งเดียว!)

**กฎเหล็ก (Week 14)**: ห้าม evaluate บน test set ระหว่างขั้นตอนเลือกโมเดล — ใช้ test set ประเมิน**ครั้งเดียว**หลังตัดสินใจเลือกโมเดลสุดท้ายเสร็จแล้วเท่านั้น เพื่อป้องกัน optimistic bias จาก data leakage

In [ ]:
# ─── Refit โมเดลที่เลือกบน Training Set ทั้งหมด แล้ว evaluate บน Test ─
# วัตถุประสงค์: รายงานผลสุดท้ายที่ไม่ถูก bias จากการดู test ระหว่าง selection
final_model_name = best_row['Model']
final_pipe = Pipeline([('scaler', StandardScaler()), ('clf', models[final_model_name])])
final_pipe.fit(X_train, y_train)

y_pred_final = final_pipe.predict(X_test)
y_proba_final = final_pipe.predict_proba(X_test)[:, 1]

print(f"=== Final Model: {final_model_name} ===")
print(classification_report(y_test, y_pred_final, target_names=['Died', 'Survived']))
print(f"Test AUC: {roc_auc_score(y_test, y_proba_final):.4f}")

# ─── Confusion Matrix Visualization ──────────────────────────────────
cm = confusion_matrix(y_test, y_pred_final)
ConfusionMatrixDisplay(cm, display_labels=['Died', 'Survived']).plot(cmap='Blues')
plt.title(f'Confusion Matrix — {final_model_name}')
plt.show()

---
## สรุปทั้งหมด: จาก EDA สู่ Model Selection

| Part | สิ่งที่ทำ | ผลลัพธ์ที่ได้ |
|------|----------|-------------|
| 1. Data Understanding | ตรวจ shape, dtypes, missing, duplicates, cardinality | รู้ว่าต้อง impute age/embarked, ตัด/flag deck |
| 2. Visualization | Univariate, bivariate, target-focused, multivariate plots | เห็น pattern เบื้องต้นว่า sex, pclass, age น่าจะมีผล |
| 3. EDA เจาะลึก | Outlier (IQR), distribution shape (skew/QQ/Shapiro), VIF | ตัดสินใจ preprocessing strategy อย่างมีหลักฐาน |
| 4. Hypothesis Testing | Chi-square, t-test/Mann-Whitney พิสูจน์สิ่งที่เห็นใน EDA | ยืนยันด้วยตัวเลขว่า feature ไหนมีผลจริง (ไม่ใช่ดูด้วยตา) |
| 5. Problem Framing | กำหนด X, Y, task type, metric, train/test split | โจทย์ชัดเจน: Binary Classification, วัดด้วย AUC+Accuracy |
| 6. Model Selection | Fit 5 methods, CV, One-SE Rule, final test evaluation | เลือกโมเดลสุดท้ายอย่างมีเหตุผล ไม่ใช่สุ่มเลือก |

**นี่คือโครงสร้างเดียวกับที่ Final Project ต้องการ** (CLO1–4) — ต่างกันแค่ว่า Final Project เพิ่ม CLO1 (Linear Algebra: Covariance/PCA) เข้าไปก่อน Part 4 ของสมุดนี้ ดูรายละเอียดได้ที่ `final_project_guidelines.md` และใช้ `lab15_project_template.ipynb` เป็นโครงร่างสำหรับกรอกงานของกลุ่มตัวเอง

## คำถามทบทวน (Reflection Questions)

1. ถ้า EDA พบว่า feature หนึ่ง (เช่น `fare`) มี correlation กับ target สูงมากใน scatter plot แต่ Hypothesis Test ให้ p-value > 0.05 เราควรเชื่อกราฟหรือเชื่อผล test? เพราะเหตุใด?

2. ถ้า dataset ของกลุ่มคุณมี target เป็นตัวเลขต่อเนื่อง (เช่น ราคาบ้าน) แทนที่จะเป็น binary — Part 5 และ Part 6 ของสมุดนี้ต้องปรับอย่างไรบ้าง (metric อะไรที่ต้องใช้แทน AUC/Accuracy, โมเดลไหนที่ต้องใช้แทน Logistic/LDA/QDA)?

3. ทำไมเราต้องทำ Hypothesis Testing (Part 4) ทั้งที่ดูจากกราฟใน Part 2 ก็เหมือนจะรู้คำตอบอยู่แล้ว?

4. ถ้า Shapiro-Wilk test บอกว่าตัวแปรหนึ่งไม่ Normal distribution แต่เราอยากใช้ LDA (ซึ่ง assume Gaussian) กับตัวแปรนั้น เราควรทำอย่างไร? มีทางเลือกอะไรบ้าง?